# 06 — Held-out validation (GoEmotions + EmoBank)

**Project:** [EmoVecLLM](https://github.com/drgzkr/EmoVecLLM) — replicating Anthropic 2026 emotion vectors on open-source LLMs.

nb04 built emotion vectors from *generated* stories; nb05/05b visualised them. This notebook asks the question that actually matters: **do those directions generalise to held-out, human-labelled text written by other people?**

Two complementary tests, both treating the emotion vectors as a **linear probe** on the target model's residual stream:

1. **GoEmotions (classification).** For each labelled snippet, embed it, project onto every emotion vector, and check whether the *matching* emotion's loading is high (top-1 / top-k over the full 171, a restricted k-way accuracy over the mappable labels, and a matched-vs-mismatched AUC).
2. **EmoBank (valence/arousal circumplex).** Anthropic's headline geometric result is that PC1 ≈ valence and PC2 ≈ arousal. We test it on held-out text: project EmoBank snippets onto the **emotion-vector principal axes** and correlate with the human V/A ratings.

Everything keys off the same `emotion_vectors.npz` nb04 wrote, and runs the **same target model** the vectors came from (so the probe lives in the vectors' own space).

## 0 — Setup

Same portable bootstrap as nb04 (Drive on Colab / `EMOVEC_WORK_DIR` elsewhere) plus `datasets` (GoEmotions), `pandas` (EmoBank CSV), and `scikit-learn` / `scipy`.

| Env var | Effect | Default |
|---|---|---|
| `EMOVEC_WORK_DIR` | Where `features/` lives. | Drive on Colab, repo root locally |
| `EMOVEC_TARGET_MODEL` | Override the probed model (else read from the vectors). | from npz |
| `EMOVEC_PROJ_LAYER` | Layer to probe at (`mid`/int). | vectors' cluster layer |
| `EMOVEC_DEMO` | Cap held-out examples for a quick pass. | `1` |
| `EMOVEC_N_HELDOUT` | Examples per dataset. | demo `300`, else `3000` |
| `EMOVEC_VECTORS_PATH` | Explicit `emotion_vectors.npz` (skip discovery). | auto |

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.40", "accelerate>=0.30", "bitsandbytes>=0.43",
                    "datasets>=2.18", "scikit-learn", "pandas", "scipy", "tqdm"], check=False)


In [ ]:
import json, os, re, time
from pathlib import Path
from collections import Counter
import numpy as np

def _env(name, default):
    v = os.environ.get(name); return v if v not in (None, "") else default
def _env_int(name, default):
    v = os.environ.get(name); return int(v) if v not in (None, "") else default
def _env_flag(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "on")

if "HF_TOKEN" not in os.environ and IN_COLAB:
    try:
        from google.colab import userdata
        _t = userdata.get("HF_TOKEN")
        if _t: os.environ["HF_TOKEN"] = _t
    except Exception:
        pass

MOUNT_DRIVE = _env_flag("EMOVEC_MOUNT_DRIVE", IN_COLAB)
if os.environ.get("EMOVEC_WORK_DIR"):
    WORK_DIR = Path(os.environ["EMOVEC_WORK_DIR"]).expanduser()
elif IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/EmoVecLLM")
elif IN_COLAB:
    WORK_DIR = Path("/content/EmoVecLLM")
else:
    WORK_DIR = Path("..").resolve()
DATA_DIR = WORK_DIR / "data" / "processed"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9) if DEVICE == "cuda" else 0.0
print(f"WORK_DIR={WORK_DIR}  device={DEVICE} ({VRAM_GB:.0f} GB)")


## 1 — Load the emotion vectors

Auto-discover `features/**/emotion_vectors.npz` (largest emotion coverage), or set `EMOVEC_VECTORS_PATH`. We read the vectors, the neutral **baseline**, the cluster layer, the **target model**, and `skip_first` — so the probe matches exactly how the vectors were built.

In [ ]:
DEMO       = _env_flag("EMOVEC_DEMO", True)
N_HELDOUT  = _env_int("EMOVEC_N_HELDOUT", 300 if DEMO else 3000)

vec_path = _env("EMOVEC_VECTORS_PATH", "")
if vec_path:
    VEC_FILE = Path(vec_path).expanduser()
else:
    cands = sorted((DATA_DIR / "features").rglob("emotion_vectors.npz"))
    assert cands, f"no emotion_vectors.npz under {DATA_DIR/'features'} — run nb04 first."
    def _ncov(p):
        try: return len(np.load(p, allow_pickle=True)["emotions"])
        except Exception: return 0
    VEC_FILE = max(cands, key=_ncov)
print(f"vectors: {VEC_FILE}")

vec = np.load(VEC_FILE, allow_pickle=True)
V          = vec["vectors"]                       # (E, n_layers, d)
EMOTIONS   = [str(e) for e in vec["emotions"]]
BASELINE   = vec["baseline_mean"]                 # (n_layers, d)
CLUSTER_L  = int(vec["cluster_layer"])
SKIP_FIRST = bool(vec["skip_first"]) if "skip_first" in vec.files else True
TARGET_MODEL = _env("EMOVEC_TARGET_MODEL", str(vec["target_model"]) if "target_model" in vec.files else "gpt2")
E, N_LAYERS, D_MODEL = V.shape

proj_spec = _env("EMOVEC_PROJ_LAYER", "")
PROJ_LAYER = (N_LAYERS // 2 if proj_spec == "mid"
              else int(proj_spec) if proj_spec else CLUSTER_L)

def _unit(v, axis=-1, eps=1e-12):
    return v / (np.linalg.norm(v, axis=axis, keepdims=True) + eps)

VHAT = _unit(V[:, PROJ_LAYER, :].astype(np.float64))   # (E, d) unit emotion directions
BASE = BASELINE[PROJ_LAYER].astype(np.float64)         # (d,)
print(f"emotions={E}  target={TARGET_MODEL}  proj_layer={PROJ_LAYER}/{N_LAYERS}  skip_first={SKIP_FIRST}")
print(f"demo={DEMO}  n_heldout/dataset={N_HELDOUT}")


## 2 — Scoring helpers (pure NumPy)

Deliberately model-free so they're unit-testable: classification metrics over the loading matrix `L (n_examples, E)`, and the circumplex alignment.

In [ ]:
def topk_accuracy(L, true_idx, k):
    'Fraction of rows whose true emotion is in the top-k loadings.'
    order = np.argsort(-L, axis=1)[:, :k]
    return float(np.mean([t in order[i] for i, t in enumerate(true_idx)]))

def restricted_accuracy(L, true_local, label_cols):
    'Argmax restricted to the mappable label columns (k-way, k=len(label_cols)).'
    sub = L[:, label_cols]
    pred = sub.argmax(axis=1)
    return float(np.mean(pred == np.asarray(true_local)))

def matched_vs_mismatched_auc(L, true_idx):
    '''Mean over rows of the fraction of OTHER emotions the true one outscores —
    an AUC-style "is the matched emotion ranked above a random mismatch?" (0.5=chance).'''
    aucs = []
    for i, t in enumerate(true_idx):
        row = L[i]; others = np.delete(row, t)
        aucs.append(float(np.mean(row[t] > others)))
    return float(np.mean(aucs))

def circumplex_alignment(F, base, M, target_a, target_b):
    '''Project held-out embeddings F (n,d) onto the top-2 PCs of emotion matrix
    M (E,d), correlate with two human axes. Returns dict with r for the best of
    the two PC->axis assignments, sign-aligned. Pearson r.'''
    from numpy.linalg import svd
    Mc = M - M.mean(0)
    _, _, Vt = svd(Mc, full_matrices=False)
    comps = Vt[:2]                                   # (2, d)
    coords = (F - base) @ comps.T                    # (n, 2)
    def _r(x, y):
        x = x - x.mean(); y = y - y.mean()
        d = (np.linalg.norm(x) * np.linalg.norm(y)) + 1e-12
        return float((x @ y) / d)
    a, b = np.asarray(target_a, float), np.asarray(target_b, float)
    # try both PC->axis assignments, keep the one with higher |r| sum
    opt1 = (abs(_r(coords[:, 0], a)), abs(_r(coords[:, 1], b)), 0)
    opt2 = (abs(_r(coords[:, 1], a)), abs(_r(coords[:, 0], b)), 1)
    pcA, pcB = (0, 1) if (opt1[0] + opt1[1]) >= (opt2[0] + opt2[1]) else (1, 0)
    return {"r_axisA": _r(coords[:, pcA], a), "r_axisB": _r(coords[:, pcB], b),
            "coords": coords, "pc_for_A": pcA, "pc_for_B": pcB}


## 3 — Load the target model + embed held-out text

Same HF loader / precision logic as nb04. `embed(text)` returns the mean-pooled residual at `PROJ_LAYER` (token-0 sink dropped when `skip_first`), and `loadings(texts)` projects a batch onto every emotion vector → `(n, E)`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

def _prec(vram):
    if DEVICE == "cpu": return "fp32"
    return "4bit" if (vram and vram < 24) else "bf16"

prec = _prec(VRAM_GB); hf_token = os.environ.get("HF_TOKEN")
tok = AutoTokenizer.from_pretrained(TARGET_MODEL, trust_remote_code=True, token=hf_token)
kw = dict(trust_remote_code=True, token=hf_token, output_hidden_states=True)
if DEVICE == "cpu":
    kw.update(torch_dtype=torch.float32)
elif prec in ("4bit",) and BitsAndBytesConfig is not None:
    kw.update(quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True), device_map="auto")
else:
    dtype = torch.bfloat16 if prec == "bf16" and torch.cuda.is_bf16_supported() else torch.float16
    kw.update(torch_dtype=dtype, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(TARGET_MODEL, **kw).eval()
INPUT_DEVICE = model.get_input_embeddings().weight.device
print(f"probing {TARGET_MODEL} (precision={prec})")

@torch.no_grad()
def embed(text):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=256)
    enc = {k: v.to(INPUT_DEVICE) for k, v in enc.items()}
    h = model(**enc).hidden_states[PROJ_LAYER + 1][0]      # (seq, d), +1 skips embeddings
    if SKIP_FIRST and h.shape[0] > 1:
        h = h[1:]
    return h.mean(0).cpu().float().numpy().astype(np.float64)

def loadings(texts):
    F = np.stack([embed(t) for t in texts])               # (n, d)
    return (F - BASE) @ VHAT.T, F                          # (n, E), (n, d)


## 4 — GoEmotions: classification

We map GoEmotions' 27 labels onto whichever of our emotion words are present (synonyms where the surface form differs), keep single-label examples whose label maps, embed them, and project. Metrics:

- **top-1 / top-5 / top-10** over all `E` emotions (hard: chance ≈ `k/E`);
- **restricted k-way accuracy** — argmax among only the mappable labels (interpretable);
- **matched-vs-mismatched AUC** — is the true emotion ranked above a random other (0.5 = chance)?

Skipped cleanly if `datasets` / the download is unavailable.

In [ ]:
# GoEmotions label -> candidate emotion words in our set (first present one wins).
GOEMO_SYN = {
    "admiration": ["admiring", "impressed"], "amusement": ["amused"],
    "anger": ["angry"], "annoyance": ["annoyed", "irritated"],
    "approval": ["approving"], "caring": ["caring", "compassionate"],
    "confusion": ["confused"], "curiosity": ["curious"], "desire": ["desiring", "longing"],
    "disappointment": ["disappointed"], "disapproval": ["disapproving"],
    "disgust": ["disgusted"], "embarrassment": ["embarrassed"],
    "excitement": ["excited"], "fear": ["afraid", "scared", "fearful"],
    "gratitude": ["grateful", "thankful"], "grief": ["grieving", "heartbroken"],
    "joy": ["joyful", "happy"], "love": ["loving"], "nervousness": ["nervous", "anxious"],
    "optimism": ["optimistic", "hopeful"], "pride": ["proud"],
    "realization": ["realising"], "relief": ["relieved"], "remorse": ["remorseful", "guilty"],
    "sadness": ["sad"], "surprise": ["surprised", "amazed", "astonished"],
}

go_results = None
try:
    from datasets import load_dataset
    ds = load_dataset("go_emotions", "simplified", split="test")
    go_names = ds.features["labels"].feature.names      # 28 incl. neutral
    emo_to_idx = {e: i for i, e in enumerate(EMOTIONS)}

    # map each GoEmotions label name -> our emotion index (if any synonym present)
    label_map = {}
    for gi, gname in enumerate(go_names):
        for cand in GOEMO_SYN.get(gname, [gname]):
            if cand in emo_to_idx:
                label_map[gi] = emo_to_idx[cand]; break

    texts, true_idx = [], []
    for row in ds:
        if len(row["labels"]) != 1:        # single-label only, for a clean test
            continue
        gi = row["labels"][0]
        if gi in label_map:
            texts.append(row["text"]); true_idx.append(label_map[gi])
        if len(texts) >= N_HELDOUT:
            break

    if texts:
        L, _ = loadings(texts)
        true_idx = np.array(true_idx)
        cols = sorted(set(true_idx.tolist()))
        col_pos = {c: j for j, c in enumerate(cols)}
        true_local = [col_pos[t] for t in true_idx]
        go_results = {
            "n": len(texts), "n_label_classes": len(cols),
            "top1": topk_accuracy(L, true_idx, 1),
            "top5": topk_accuracy(L, true_idx, 5),
            "top10": topk_accuracy(L, true_idx, 10),
            "restricted_acc": restricted_accuracy(L, true_local, cols),
            "auc": matched_vs_mismatched_auc(L, true_idx),
        }
        print(json.dumps({k: round(v, 3) if isinstance(v, float) else v
                          for k, v in go_results.items()}, indent=2))
        print(f"chance: top1≈{1/E:.3f}  top10≈{10/E:.3f}  "
              f"restricted≈{1/len(cols):.3f}  auc=0.5")
    else:
        print("no mappable single-label GoEmotions examples for this emotion set.")
except Exception as ex:
    print(f"GoEmotions skipped ({type(ex).__name__}: {ex})")


## 5 — EmoBank: valence / arousal circumplex

EmoBank gives continuous **valence** and **arousal** ratings per sentence. We project held-out sentences onto the **top-2 principal axes of the emotion-vector matrix** and correlate with the human ratings — the held-out version of Anthropic's PC1↔valence (*r*≈0.81) / PC2↔arousal (*r*≈0.66) result. EmoBank is pulled as a CSV from its GitHub mirror (no `datasets` dependency); skipped cleanly if unreachable.

In [ ]:
emobank_results = None
try:
    import pandas as pd
    EMOBANK_URL = ("https://raw.githubusercontent.com/JULIELab/EmoBank/master/"
                   "corpus/emobank.csv")
    eb = pd.read_csv(_env("EMOVEC_EMOBANK_CSV", EMOBANK_URL))
    if "split" in eb.columns:
        eb = eb[eb["split"] == "test"]
    eb = eb.dropna(subset=["text", "V", "A"]).head(N_HELDOUT)
    texts = eb["text"].astype(str).tolist()
    valence = eb["V"].to_numpy(float)
    arousal = eb["A"].to_numpy(float)

    _, F = loadings(texts)                                  # (n, d) embeddings
    al = circumplex_alignment(F, BASE, V[:, PROJ_LAYER, :].astype(np.float64),
                              valence, arousal)
    emobank_results = {"n": len(texts),
                       "r_valence": al["r_axisA"], "r_arousal": al["r_axisB"],
                       "pc_for_valence": al["pc_for_A"], "pc_for_arousal": al["pc_for_B"]}
    print(json.dumps({k: round(v, 3) if isinstance(v, float) else v
                      for k, v in emobank_results.items()}, indent=2))
    print("Anthropic (Claude Sonnet 4.5): r_valence≈0.81  r_arousal≈0.66")

    import matplotlib.pyplot as plt
    coords = al["coords"]
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for k, (axis_vals, name, pc) in enumerate(
            [(valence, "valence", al["pc_for_A"]), (arousal, "arousal", al["pc_for_B"])]):
        sc = ax[k].scatter(coords[:, pc], axis_vals, s=6, alpha=0.4)
        ax[k].set_xlabel(f"emotion-vector PC{pc+1} (held-out projection)")
        ax[k].set_ylabel(f"EmoBank {name}")
        r = emobank_results["r_valence"] if k == 0 else emobank_results["r_arousal"]
        ax[k].set_title(f"{name}: r = {r:.2f}")
    plt.tight_layout(); plt.show()
except Exception as ex:
    print(f"EmoBank skipped ({type(ex).__name__}: {ex})")


## 6 — Save metrics + interpretation

Both results are written to `features/.../validation.json` next to the vectors, so a model-comparison sweep (nb10) can collect them across targets. Read the numbers honestly: on a **demo** run (tiny target like `gpt2`, partial story coverage) expect weak-but-above-chance scores — the point is the harness. The science run wants a real target (pythia-1.4b → 7–8B) and full coverage; that's when these should approach the paper's numbers.

In [ ]:
summary = {"vectors_file": str(VEC_FILE), "target_model": TARGET_MODEL,
           "proj_layer": PROJ_LAYER, "n_emotions": E, "demo": DEMO,
           "goemotions": go_results, "emobank": emobank_results,
           "updated": time.strftime("%Y-%m-%dT%H:%M:%S")}
out = VEC_FILE.parent / "validation.json"
out.write_text(json.dumps(summary, indent=2, default=lambda o: float(o)))
print(f"saved -> {out}")
print()
if go_results:
    print(f"GoEmotions: top10={go_results['top10']:.2f} (chance {10/E:.2f}), "
          f"restricted {go_results['n_label_classes']}-way={go_results['restricted_acc']:.2f}, "
          f"AUC={go_results['auc']:.2f}")
if emobank_results:
    print(f"EmoBank: r_valence={emobank_results['r_valence']:.2f}, "
          f"r_arousal={emobank_results['r_arousal']:.2f}")
if not (go_results or emobank_results):
    print("No external dataset reachable — re-run with internet for the real validation.")
